In [1]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

from drGAT.metrics import get_parsed_df

In [3]:
def extract_method_data(filename):
    match = re.match(r"([A-Za-zA-Z0-9]+)_([a-z0-9]+)\.sqlite3", filename)
    return match.groups() if match else ("unknown", "unknown")

In [4]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ Transformer_gdsc1.sqlite3: 340 trials completed
✅ GAT_gdsc1.sqlite3: 247 trials completed
✅ GAT_ctrp.sqlite3: 269 trials completed
✅ GAT_gdsc2.sqlite3: 230 trials completed
✅ GAT_nci.sqlite3: 1054 trials completed
✅ Transformer_nci.sqlite3: 994 trials completed
✅ Transformer_ctrp.sqlite3: 189 trials completed
✅ GATv2_ctrp.sqlite3: 215 trials completed
✅ GATv2_nci.sqlite3: 1308 trials completed
✅ GATv2_gdsc2.sqlite3: 203 trials completed
✅ GATv2_gdsc1.sqlite3: 229 trials completed
✅ Transformer_gdsc2.sqlite3: 349 trials completed


n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                1054  0.876 (± 0.007)  0.914 (± 0.008)   
      GATv2              1308  0.875 (± 0.006)  0.915 (± 0.014)   
      Transformer         994  0.868 (± 0.009)  0.915 (± 0.011)   
gdsc1 GAT                 247   0.79 (± 0.007)   0.787 (± 0.01)   
      GATv2               229  0.778 (± 0.011)  0.779 (± 0.013)   
      Transformer         340  0.803 (± 0.004)  0.794 (± 0.012)   
gdsc2 GAT                 230  0.801 (± 0.011)  0.801 (± 0.019)   
      GATv2               203   0.803 (± 0.01)  0.806 (± 0.017)   
      Transformer         349  0.819 (± 0.012)  0.815 (± 0.024)   
ctrp  GAT                 269  0.842 (± 0.004)  0.852 (± 0.006)   
      GATv2               215  0.839 (± 0.006)  0.849 (± 0.005)   
      Transformer         189   0.85 (± 0.005)  0.853 (± 0.007)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.826 (± 0.012)  0.868 (± 0.007)  0.948 (± 0.003)   
      GATv2        0.824 (± 0.017)  0.867 (± 0.006)  0.948 (± 0.003)   
      Transformer  0.807 (± 0.023)  0.857 (± 0.012)  0.945 (± 0.003)   
gdsc1 GAT           0.78 (± 0.014)  0.784 (± 0.006)  0.867 (± 0.007)   
      GATv2         0.76 (± 0.013)  0.769 (± 0.011)  0.858 (± 0.009)   
      Transformer  0.805 (± 0.009)  0.799 (± 0.004)  0.884 (± 0.002)   
gdsc2 GAT           0.82 (± 0.007)   0.81 (± 0.011)  0.877 (± 0.012)   
      GATv2        0.816 (± 0.016)  0.811 (± 0.008)   0.88 (± 0.009)   
      Transformer   0.843 (± 0.02)  0.828 (± 0.011)  0.895 (± 0.011)   
ctrp  GAT          0.844 (± 0.011)  0.848 (± 0.003)  0.919 (± 0.003)   
      GATv2          0.84 (± 0.01)  0.844 (± 0.006)  0.919 (± 0.004)   
      Transformer  0.861 (± 0.006)  0.857 (± 0.004)  0.928 (± 0.003)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.946 (± 0.004)  0.875 (± 0.007)           NaN   
      GATv2        0.947 (± 0.003)  0.875 (± 0.006)           NaN   
      Transformer  0.943 (± 0.004)   0.867 (± 0.01)           NaN   
gdsc1 GAT          0.869 (± 0.007)   0.79 (± 0.007)           NaN   
      GATv2         0.861 (± 0.01)  0.778 (± 0.011)           NaN   
      Transformer  0.885 (± 0.004)  0.803 (± 0.004)           NaN   
gdsc2 GAT          0.876 (± 0.018)    0.8 (± 0.011)           NaN   
      GATv2         0.88 (± 0.013)  0.802 (± 0.011)           NaN   
      Transformer  0.899 (± 0.016)  0.818 (± 0.012)           NaN   
ctrp  GAT          0.927 (± 0.003)  0.842 (± 0.004)           NaN   
      GATv2        0.928 (± 0.004)  0.839 (± 0.005)           NaN   
      Transformer  0.936 (± 0.004)   0.85 (± 0.005)           NaN   

                  params_activation  ...  params_hidden1  params_hidden2  \
data  method                         ...                                   
nci   GAT                      relu  ...             259              96   
      GATv2                    relu  ...             395             236   
      Transformer              relu  ...             289             215   
gdsc1 GAT                      gelu  ...             325             236   
      GATv2                    gelu  ...             362              85   
      Transformer              gelu  ...             283             134   
gdsc2 GAT                      gelu  ...             496             173   
      GATv2                    relu  ...             436             123   
      Transformer              relu  ...             432             198   
ctrp  GAT                      gelu  ...             398              83   
      GATv2                    gelu  ...             436              66   
      Transformer              relu  ...             477             153   

                   params_hidden3  para

In [5]:
best_df[desired_order]

n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                1054  0.876 (± 0.007)  0.914 (± 0.008)   
      GATv2              1308  0.875 (± 0.006)  0.915 (± 0.014)   
      Transformer         994  0.868 (± 0.009)  0.915 (± 0.011)   
gdsc1 GAT                 247   0.79 (± 0.007)   0.787 (± 0.01)   
      GATv2               229  0.778 (± 0.011)  0.779 (± 0.013)   
      Transformer         340  0.803 (± 0.004)  0.794 (± 0.012)   
gdsc2 GAT                 230  0.801 (± 0.011)  0.801 (± 0.019)   
      GATv2               203   0.803 (± 0.01)  0.806 (± 0.017)   
      Transformer         349  0.819 (± 0.012)  0.815 (± 0.024)   
ctrp  GAT                 269  0.842 (± 0.004)  0.852 (± 0.006)   
      GATv2               215  0.839 (± 0.006)  0.849 (± 0.005)   
      Transformer         189   0.85 (± 0.005)  0.853 (± 0.007)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.826 (± 0.012)  0.868 (± 0.007)  0.948 (± 0.003)   
      GATv2        0.824 (± 0.017)  0.867 (± 0.006)  0.948 (± 0.003)   
      Transformer  0.807 (± 0.023)  0.857 (± 0.012)  0.945 (± 0.003)   
gdsc1 GAT           0.78 (± 0.014)  0.784 (± 0.006)  0.867 (± 0.007)   
      GATv2         0.76 (± 0.013)  0.769 (± 0.011)  0.858 (± 0.009)   
      Transformer  0.805 (± 0.009)  0.799 (± 0.004)  0.884 (± 0.002)   
gdsc2 GAT           0.82 (± 0.007)   0.81 (± 0.011)  0.877 (± 0.012)   
      GATv2        0.816 (± 0.016)  0.811 (± 0.008)   0.88 (± 0.009)   
      Transformer   0.843 (± 0.02)  0.828 (± 0.011)  0.895 (± 0.011)   
ctrp  GAT          0.844 (± 0.011)  0.848 (± 0.003)  0.919 (± 0.003)   
      GATv2          0.84 (± 0.01)  0.844 (± 0.006)  0.919 (± 0.004)   
      Transformer  0.861 (± 0.006)  0.857 (± 0.004)  0.928 (± 0.003)   

                              AUPR  
data  method                        
nci   GAT          0.946 (± 0.004)  
      GATv2        0.947 (± 0.003)  
      Transformer  0.943 (± 0.004)  
gdsc1 GAT          0.869 (± 0.007)  
      GATv2         0.861 (± 0.01)  
      Transformer  0.885 (± 0.004)  
gdsc2 GAT          0.876 (± 0.018)  
      GATv2         0.88 (± 0.013)  
      Transformer  0.899 (± 0.016)  
ctrp  GAT          0.927 (± 0.003)  
      GATv2        0.928 (± 0.004)  
      Transformer  0.936 (± 0.004)

In [6]:
[i for i in df_all.columns if 'params' in i]

['params_T_max',
 'params_activation',
 'params_attention_dropout',
 'params_dropout1',
 'params_dropout2',
 'params_dropout3',
 'params_epochs',
 'params_final_mlp_layers',
 'params_heads',
 'params_hidden1',
 'params_hidden2',
 'params_hidden3',
 'params_is_zero_pad',
 'params_lr',
 'params_n_layers',
 'params_norm_type',
 'params_optimizer',
 'params_scheduler',
 'params_weight_decay']